[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Models and Fields &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup with the models its worked examples wrote: `CatalogModel`,
`Author`, `Book` and `Payment`, in a database in memory with the catalog loaded. Run it first. The
tasks build on one another, so run them in order.


In [1]:
import datetime
import re
import subprocess
import sys
from decimal import Decimal
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (BooleanField, CharField, CompositeKey, DateField, DateTimeField, DecimalField,
                    ForeignKeyField, IntegerField, Model, SqliteDatabase, TextField)

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

def message(error):
    """An error's class and text, without the memory address that makes no two runs agree."""
    return f"{type(error).__module__}.{type(error).__name__}: {re.sub(r'0x[0-9a-f]+', '0x...', str(error))}"

def table_sql(model):
    """The CREATE TABLE peewee writes for a model, and any index that goes with it."""
    written = [model._schema._create_table().query()[0]]
    return "\n".join(written + [index.query()[0] for index in model._schema._create_indexes()])

db = SqliteDatabase(":memory:")

print("peewee", peewee.__version__, "| the database:", db.database, "| open:", not db.is_closed())

class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()


class Payment(CatalogModel):
    amount = DecimalField(max_digits=8, decimal_places=3)
    at = DateTimeField()


def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()

build(db)
db.create_tables([Payment])

print("loaded:", Author.select().count(), "authors and", Book.select().count(), "books")


peewee 4.5.1 | the database: :memory: | open: False
loaded: 4 authors and 12 books


**1.** A shelf, with one of each kind of rule.


In [2]:
class Shelf(CatalogModel):
    code = CharField(max_length=10, unique=True)
    floor = IntegerField(index=True)
    note = CharField(max_length=80, null=True)


print(table_sql(Shelf))


CREATE TABLE IF NOT EXISTS "shelf" ("id" INTEGER NOT NULL PRIMARY KEY, "code" VARCHAR(10) NOT NULL, "floor" INTEGER NOT NULL, "note" VARCHAR(80))
CREATE UNIQUE INDEX IF NOT EXISTS "shelf_code" ON "shelf" ("code")
CREATE INDEX IF NOT EXISTS "shelf_floor" ON "shelf" ("floor")


Three rules and two statements beside the table: the unique one for the code, the plain one for the
floor. `null=True` is the only one of the three that is inside the `CREATE TABLE`, as the absence of
a `NOT NULL`.


**2.** The same code twice.


In [3]:
db.create_tables([Shelf])
Shelf.create(code="A-1", floor=1)
Shelf.create(code="A-2", floor=1, note="by the window")
print("shelves:", [shelf.code for shelf in Shelf.select().order_by(Shelf.code)])

try:
    Shelf.create(code="A-1", floor=2)
except peewee.IntegrityError as error:
    print(message(error))


shelves: ['A-1', 'A-2']
peewee.IntegrityError: UNIQUE constraint failed: shelf.code


The refusal names the index rather than the column, which is what a unique index is called when it
is broken.


**3.** A model whose table is called something else.


In [4]:
class Copy(CatalogModel):
    book = ForeignKeyField(Book, backref="copies")
    shelf = ForeignKeyField(Shelf, backref="copies")
    acquired = DateField()

    class Meta:
        table_name = "stock"


print("the class is", Copy.__name__, "and the table is", Copy._meta.table_name)
print(table_sql(Copy).splitlines()[0])


the class is Copy and the table is stock
CREATE TABLE IF NOT EXISTS "stock" ("id" INTEGER NOT NULL PRIMARY KEY, "book_id" INTEGER NOT NULL, "shelf_id" INTEGER NOT NULL, "acquired" DATE NOT NULL, FOREIGN KEY ("book_id") REFERENCES "book" ("id"), FOREIGN KEY ("shelf_id") REFERENCES "shelf" ("id"))


`Meta.table_name` is what to reach for against a database somebody else designed, where the table
is `stock` and renaming it is not on offer.


**4.** One copy of a book to a shelf.


In [5]:
class Copy(CatalogModel):
    book = ForeignKeyField(Book, backref="copies")
    shelf = ForeignKeyField(Shelf, backref="copies")
    acquired = DateField()

    class Meta:
        table_name = "stock"
        indexes = ((("book", "shelf"), True),)                      # one copy of a book to a shelf


db.create_tables([Copy])
print(table_sql(Copy).splitlines()[-1])

salt_road = Book.get(Book.title == "The Salt Road")
Copy.create(book=salt_road, shelf=1, acquired=datetime.date(2026, 1, 5))
Copy.create(book=salt_road, shelf=2, acquired=datetime.date(2026, 1, 5))    # another shelf, fine
try:
    Copy.create(book=salt_road, shelf=1, acquired=datetime.date(2026, 2, 1))
except peewee.IntegrityError as error:
    print(message(error))


CREATE UNIQUE INDEX IF NOT EXISTS "copy_book_id_shelf_id" ON "stock" ("book_id", "shelf_id")
peewee.IntegrityError: UNIQUE constraint failed: stock.book_id, stock.shelf_id


The columns in the message are `book_id` and `shelf_id`, which are the columns a foreign key makes,
rather than the attribute names in the model.


**5.** A value written by something that is not the model.


In [6]:
db.execute_sql("INSERT INTO payment (amount, at) VALUES (?, ?)", ("3.750", "sometime on Tuesday"))

written = Payment.get_by_id(1)
print("amount:", repr(written.amount), "| at:", repr(written.at))


amount: Decimal('3.75') | at: 'sometime on Tuesday'


The amount came back a `Decimal`, because the text SQLite holds is a number peewee's field knows how
to read. The time did not, because `sometime on Tuesday` is not a date and the field has nothing to
convert: what comes back is the text that is in the column.


**6.** A connection opened for one block.


In [7]:
print("before the block:", not db.is_closed())
with db.connection_context():
    print("inside          :", Book.select().count(), "books | open:", not db.is_closed())
print("after the block :", not db.is_closed())


before the block: True
inside          : 12 books | open: True
after the block : False


The block opened a connection and closed it afterwards. Everything before it worked because
`autoconnect` is on, which opened one on the first query and left it open.


---

&#8592; **Back to:** [Models and Fields](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/02-models-and-fields.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
